In [2]:
# ─────────────────────────────────────────────────────────────
# CELL 17 — Load final model for validation
# ─────────────────────────────────────────────────────────────

import torch
import sentencepiece as spm
from transformers import LlamaForCausalLM, LlamaConfig

FINAL_MODEL_DIR = "./BanglaLLM/final_model"
SP_MODEL_PATH   = "./BanglaLLM/tokenizer/bangla_spm.model"   # ← hardcoded
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"
# ── Re-declare paths (run this if starting a fresh kernel) ───
TOKENIZER_OUTPUT = "./BanglaLLM/tokenizer"
FINAL_MODEL_DIR  = "./BanglaLLM/final_model"
CLEANED_OUTPUT   = "./BanglaLLM/cleaned"
FINAL_OUTPUT     = "./BanglaLLM/final_dataset"
CHUNK_FILE       = f"{FINAL_OUTPUT}/chunks.arrow"
MAX_LENGTH       = 2048

# Load SP tokenizer
sp = spm.SentencePieceProcessor(model_file=SP_MODEL_PATH)

# Load trained model
config = LlamaConfig.from_pretrained(FINAL_MODEL_DIR)
model  = LlamaForCausalLM.from_pretrained(
    FINAL_MODEL_DIR,
    torch_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
)
model.to(DEVICE)
model.eval()

print(f"Model loaded from  : {FINAL_MODEL_DIR}")
print(f"Device             : {DEVICE}")
print(f"Parameters         : {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")
print(f"Vocab size         : {sp.get_piece_size():,}")

Loading weights: 100%|██████████| 75/75 [00:00<00:00, 491.64it/s]


Model loaded from  : ./BanglaLLM/final_model
Device             : cuda
Parameters         : 51.7M
Vocab size         : 32,000


In [4]:
# ─────────────────────────────────────────────────────────────
# CELL 18 — Perplexity on eval set
# Lower = better. Untrained model: ~32000 (random)
# Good Bengali model: < 50
# ─────────────────────────────────────────────────────────────
# ── Re-build eval_torch (run this if starting a fresh kernel) ─
import pyarrow as pa
import pyarrow.ipc as ipc
import torch
from datasets import Dataset
from torch.utils.data import Dataset as TorchDataset

CHUNK_FILE = "./BanglaLLM/final_dataset/chunks.arrow"
MAX_LENGTH = 2048

print("Loading chunked dataset...")
source          = pa.memory_map(CHUNK_FILE, "r")
table           = ipc.open_file(source).read_all()
chunked_dataset = Dataset(table)

split         = chunked_dataset.train_test_split(test_size=0.05, seed=42)
eval_dataset  = split["test"]

class BengaliCLMDataset(TorchDataset):
    def __init__(self, hf_dataset):
        self.data = hf_dataset

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        ids = torch.tensor(self.data[idx]["input_ids"], dtype=torch.long)
        return {"input_ids": ids, "labels": ids.clone()}

eval_torch = BengaliCLMDataset(eval_dataset)

print(f"Eval chunks ready : {len(eval_torch):,}")
print(f"Sample shape      : {eval_torch[0]['input_ids'].shape}")

import math
import torch
from torch.utils.data import DataLoader

NUM_EVAL_BATCHES = 100     # how many batches to evaluate (None = full eval set)

eval_loader = DataLoader(
    eval_torch,
    batch_size  = 4,
    shuffle     = False,
    pin_memory  = True,
)

total_loss  = 0.0
total_steps = 0

print("Computing perplexity on eval set...")

with torch.no_grad():
    for i, batch in enumerate(eval_loader):
        if NUM_EVAL_BATCHES and i >= NUM_EVAL_BATCHES:
            break

        input_ids = batch["input_ids"].to(DEVICE)
        labels    = batch["labels"].to(DEVICE)

        outputs = model(input_ids=input_ids, labels=labels)
        total_loss  += outputs.loss.item()
        total_steps += 1

        if (i + 1) % 20 == 0:
            running_ppl = math.exp(total_loss / total_steps)
            print(f"  Batch {i+1:>4} | running perplexity: {running_ppl:.2f}", end="\r")

avg_loss    = total_loss / total_steps
perplexity  = math.exp(avg_loss)

print(f"\n{'─' * 50}")
print(f"Batches evaluated : {total_steps:,}")
print(f"Average loss      : {avg_loss:.4f}")
print(f"Perplexity        : {perplexity:.2f}")
print(f"{'─' * 50}")

if perplexity < 20:
    print("Excellent — model has learned Bengali well")
elif perplexity < 50:
    print("Good — model shows solid Bengali understanding")
elif perplexity < 100:
    print("Moderate — model is learning but needs more data/epochs")
else:
    print("High perplexity — model needs more training")

Loading chunked dataset...
Eval chunks ready : 476
Sample shape      : torch.Size([2048])
Computing perplexity on eval set...
  Batch  100 | running perplexity: 351.42
──────────────────────────────────────────────────
Batches evaluated : 100
Average loss      : 5.8620
Perplexity        : 351.42
──────────────────────────────────────────────────
High perplexity — model needs more training


In [5]:
# ─────────────────────────────────────────────────────────────
# CELL 19 — Text generation quality check
# Tests whether model produces coherent Bengali continuations
# ─────────────────────────────────────────────────────────────

def generate(prompt, max_new_tokens=80, temperature=0.8, top_p=0.9, top_k=50):
    ids       = [sp.bos_id()] + sp.encode(prompt)
    input_ids = torch.tensor([ids], dtype=torch.long).to(DEVICE)

    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens = max_new_tokens,
            do_sample      = True,
            temperature    = temperature,
            top_p          = top_p,
            top_k          = top_k,
            eos_token_id   = sp.eos_id(),
            pad_token_id   = 0,
            repetition_penalty = 1.2,   # penalise repeated tokens
        )

    # Decode only the newly generated tokens
    new_ids   = output[0][len(ids):].tolist()
    generated = sp.decode(new_ids)
    return generated


# Test prompts — covers different domains in your training data
test_prompts = [
    "বাংলাদেশের ইতিহাস",
    "আমার প্রিয় খাবার হলো",
    "বিজ্ঞান ও প্রযুক্তির",
    "ঢাকা শহরে",
    "শিক্ষার গুরুত্ব হলো",
]

print("=" * 65)
print("GENERATION QUALITY TEST")
print("=" * 65)

for prompt in test_prompts:
    generated = generate(prompt)
    print(f"\nPrompt    : {prompt}")
    print(f"Generated : {generated}")
    print("─" * 65)

GENERATION QUALITY TEST

Prompt    : বাংলাদেশের ইতিহাস
Generated : ে। এই দেশের মানুষ আর মানুষের মাঝে কোন দেশ বা ব্যক্তি আছে তা আমরা যদি আমাদের জীবনের সব জায়গায় থাকতে চাই তাহলে একটা সমস্যা আছে, তাদের জন্য কিছু ভুল হবে। অনেক অনেক বিষয় হচ্ছে। ফলে আমাদের মানুষের ওপরই কোনো ধরনের সম্পর্ক নেই। তাই আমাদের দেশে যারা অর্থনৈতিক প্রভাব না দেয়া-স্বের কারণে কেউ আমাদের মতো ভালো ছিল না। কিন্তু আমি যদি কাউকে নিজের ব্যক্তিগত সমস্যার কারণে না হয় তবে তারা জানেন
─────────────────────────────────────────────────────────────────

Prompt    : আমার প্রিয় খাবার হলো
Generated : , এ জন্য। এ সময় উপস্থিত ছিলেন ও আপনার কাছে একটা লেখাটা নিয়ে এলো। তিনি বলেন, আমি তো জানি না, কিন্তু কোন কথা বলতে চায় না। আমরা যদি যে কাজ করতে পারছি তাহলে আমাদের দেশে যে কোন ধরনের কিছু হবে না। সে কি হবে? আপনিও কেন! আমি বিশ্বাস করি, তার জীবনে যারা কথা বলেছি তা বলে আছে। যখন তখন আমাকে বুঝতে হয়। সেখানে যাওয়ার পর থেকেই তাকে
─────────────────────────────────────────────────────────────────

Prompt    : বিজ্ঞান ও প্রযুক্তির
Generated : 

In [6]:
# ─────────────────────────────────────────────────────────────
# CELL 20 — Tokenizer coverage check
# Verifies SP tokenizer handles Bengali properly
# ─────────────────────────────────────────────────────────────

test_texts = [
    "আমার সোনার বাংলা, আমি তোমায় ভালোবাসি।",   # standard sentence
    "বাংলাদেশ স্বাধীন হয়েছিল ১৯৭১ সালে।",       # with digits
    "ক্ষমা করবেন, আপনি কি বাংলায় কথা বলেন?",     # conjuncts (যুক্তাক্ষর)
    "তাপমাত্রা ছিল ৩৭.৫ ডিগ্রি সেলসিয়াস।",      # decimal numbers
]

print("TOKENIZER COVERAGE CHECK")
print("=" * 65)

for text in test_texts:
    tokens   = sp.encode(text, out_type=str)
    ids      = sp.encode(text)
    decoded  = sp.decode(ids)
    is_lossless = (decoded.strip() == text.strip())

    print(f"\nText     : {text}")
    print(f"Tokens   : {tokens}")
    print(f"Count    : {len(tokens)} tokens for {len(text)} chars  "
          f"({len(text)/len(tokens):.1f} chars/token)")
    print(f"Lossless : {'✓' if is_lossless else '✗ MISMATCH — check training data'}")

print("\n" + "=" * 65)
print("Chars/token > 2.0 is good for Bengali (LLaMA default was ~0.8)")

TOKENIZER COVERAGE CHECK

Text     : আমার সোনার বাংলা, আমি তোমায় ভালোবাসি।
Tokens   : ['▁আমার', '▁সোনার', '▁বাংলা', ',', '▁আমি', '▁তোমায়', '▁ভালোবাসি', '।']
Count    : 8 tokens for 38 chars  (4.8 chars/token)
Lossless : ✓

Text     : বাংলাদেশ স্বাধীন হয়েছিল ১৯৭১ সালে।
Tokens   : ['▁বাংলাদেশ', '▁স্বাধীন', '▁হয়েছিল', '▁১৯৭১', '▁সালে', '।']
Count    : 6 tokens for 35 chars  (5.8 chars/token)
Lossless : ✓

Text     : ক্ষমা করবেন, আপনি কি বাংলায় কথা বলেন?
Tokens   : ['▁ক্ষমা', '▁করবেন', ',', '▁আপনি', '▁কি', '▁বাংলায়', '▁কথা', '▁বলেন', '?']
Count    : 9 tokens for 38 chars  (4.2 chars/token)
Lossless : ✓

Text     : তাপমাত্রা ছিল ৩৭.৫ ডিগ্রি সেলসিয়াস।
Tokens   : ['▁তাপমাত্রা', '▁ছিল', '▁৩৭', '.', '৫', '▁ডিগ্রি', '▁সেলসিয়াস', '।']
Count    : 8 tokens for 36 chars  (4.5 chars/token)
Lossless : ✓

Chars/token > 2.0 is good for Bengali (LLaMA default was ~0.8)


In [7]:
# ─────────────────────────────────────────────────────────────
# CELL 21 — Summary report
# ─────────────────────────────────────────────────────────────

print("=" * 65)
print("BANGLA LLM VALIDATION SUMMARY")
print("=" * 65)
print(f"Model dir      : {FINAL_MODEL_DIR}")
print(f"Parameters     : {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")
print(f"Vocab size     : {sp.get_piece_size():,}")
print(f"Perplexity     : {perplexity:.2f}  (lower is better)")
print(f"Avg loss       : {avg_loss:.4f}")
print(f"Eval batches   : {total_steps:,}")
print("=" * 65)

# Guidance for next steps
print("\nNext steps:")
if perplexity < 50:
    print("  → Model is ready. Consider scaling up to 300M params on A100.")
elif perplexity < 100:
    print("  → Run 2-3 more epochs or add more training data.")
else:
    print("  → Perplexity is high. Check data quality and increase training.")

BANGLA LLM VALIDATION SUMMARY
Model dir      : ./BanglaLLM/final_model
Parameters     : 51.7M
Vocab size     : 32,000
Perplexity     : 351.42  (lower is better)
Avg loss       : 5.8620
Eval batches   : 100

Next steps:
  → Perplexity is high. Check data quality and increase training.
